**Setup:** download `data/zip/fintech.zip` from the DA Launchpad Datasets page and unzip it in the same folder as this notebook, so that paths like `fintech/<table>.csv` work. Needs pandas (and matplotlib/seaborn for charts).

# Matplotlib and seaborn gallery

This notebook draws eight charts on the fintech and ecommerce data. Before each chart, a short note says which chart type to pick and why.

Put the CSV files for this dataset in a folder named `fintech and ecommerce` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

tx = pd.read_csv("fintech/transactions.csv").drop_duplicates(subset="txn_id").copy()
tx["amount_inr"] = tx["amount"].str.replace(r"[₹,\s]", "", regex=True).astype(float)
tx["ts"] = pd.to_datetime(tx["txn_datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce").combine_first(
    pd.to_datetime(tx["txn_datetime"], format="%d-%m-%Y %H:%M:%S", errors="coerce"))
tx["hour"] = tx["ts"].dt.hour
tx["failed"] = tx["status"].eq("FAILED")

## Chart 1

Chart chooser: compare a value across categories. Use a horizontal bar chart when the labels are long.

In [ ]:
rate = tx.groupby("bank")["failed"].mean().mul(100).sort_values()
ax = rate.plot(kind="barh", figsize=(8, 4), color="#4C72B0")
ax.set_xlabel("Failure rate (%)")
ax.set_title("Failure rate by bank")
plt.tight_layout()
plt.show()

## Chart 2

Chart chooser: show a trend over time. Use a line chart with one point per month.

In [ ]:
monthly = tx.groupby(tx["ts"].dt.to_period("M")).size()
ax = monthly.plot(figsize=(8, 4), marker="o")
ax.set_ylabel("Transactions")
ax.set_title("Transactions per month")
plt.tight_layout()
plt.show()

## Chart 3

Chart chooser: show the spread of one number. Use a histogram. Amounts have a long tail, so use a log scale.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(tx["amount_inr"], bins=60, log_scale=True, ax=ax)
ax.set_title("Distribution of transaction amounts (log scale)")
plt.tight_layout()
plt.show()

## Chart 4

Chart chooser: compare the spread of a number between groups. Use a box plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tx, x="txn_type", y="amount_inr", showfliers=False, ax=ax)
ax.set_ylabel("Amount (INR)")
ax.set_title("Amount by transaction type (outliers hidden)")
plt.tight_layout()
plt.show()

## Chart 5

Chart chooser: a grid of values for two categories. Use a heatmap.

In [ ]:
peak_grid = tx.pivot_table(index="payment_app", columns="hour", values="failed", aggfunc="mean").mul(100)
fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(peak_grid, cmap="Reds", annot=False, cbar_kws={"label": "Failure rate (%)"}, ax=ax)
ax.set_title("Failure rate by app and hour")
plt.tight_layout()
plt.show()

## Chart 6

Chart chooser: show the share of a total over time. Use a column chart for each month.

In [ ]:
orders = pd.read_csv("ecommerce/orders.csv").drop_duplicates(subset="order_id")
orders["order_dt"] = pd.to_datetime(orders["order_date"], format="%Y-%m-%d", errors="coerce").combine_first(
    pd.to_datetime(orders["order_date"], format="%d-%m-%Y", errors="coerce"))
sales = orders[orders["order_status"] == "Delivered"].groupby(orders["order_dt"].dt.to_period("M"))["total_amount"].sum()
ax = sales.plot(kind="bar", figsize=(10, 4), color="#55A868")
ax.set_ylabel("Revenue (INR)")
ax.set_title("Delivered revenue per month")
plt.tight_layout()
plt.show()

## Chart 7

Chart chooser: show two numbers for each item. Use a scatter plot. Add a trend line with seaborn.

In [ ]:
dl = pd.read_csv("food_delivery/deliveries.csv").drop_duplicates(subset="delivery_id")
dl = dl[dl["delivery_status"] == "Delivered"].dropna(subset=["distance_km", "actual_minutes"])
fig, ax = plt.subplots(figsize=(8, 4))
sns.regplot(data=dl, x="distance_km", y="actual_minutes", scatter_kws={"s": 5, "alpha": 0.3}, ax=ax)
ax.set_title("Delivery time against distance")
plt.tight_layout()
plt.show()

## Chart 8

Chart chooser: parts of a whole with few parts. Use a horizontal bar chart, not a pie chart, when there are more than three parts.

In [ ]:
returns = pd.read_csv("ecommerce/returns.csv")
reasons = returns["return_reason"].value_counts()
ax = reasons.plot(kind="barh", figsize=(8, 4), color="#C44E52")
ax.set_xlabel("Number of returns")
ax.set_title("Return reasons")
plt.tight_layout()
plt.show()